# MedNorm-VI E4 Tiny-Overfit Diagnostic

Intended environment: Google Colab with a GPU runtime and Google Drive mounted.
This notebook is committed **disabled and not authorized**: it will not train
until an operator sets both the run flag and the exact authorization string.

## What this decides

The completed E4 full run (12 epochs, 405,912 backward passes, 50,748 optimizer
steps) predicted **zero** mentions against 1,991 gold mentions on governed
validation. Audit 0043 proved, with no model in the loop, that the gold-grid
round-trip is exact (P = R = F1 = 1.0 on both governed splits) and that one label
ordering is shared by the target builder, the classifier output index, the
cross-entropy target and the decoder.

What is still unproven is what the trained classifier actually emits. This
notebook settles the remaining wiring question with a bounded experiment: train
on 12 governed training examples and evaluate on the very same 12.

| observed | conclusion |
| --- | --- |
| exact train F1 reaches ~1.0 | target / loss / decoder are coherent; the full-run collapse is an optimization or class-imbalance problem on the real corpus |
| training loss reaches ~0 while exact train F1 stays 0.0 | the loss is being minimized without producing decodable mentions — a target/loss/decoder inconsistency |
| neither, within the epoch cap | inconclusive; it must not be reported as a root cause |

## Three metrics, never one

A model that predicts background everywhere already scores about **0.998 grid
cell accuracy** on this corpus. Grid accuracy alone would therefore call a total
failure a near-perfect result. Every evaluation reports
`grid_cell_accuracy`, `positive_cell_accuracy` and decoded `exact_f1`
separately.

## What this is not

Not a quality result. Not a deployable checkpoint. Never an initializer for a
full E4 run. Evaluating on the training subset is the entire point of the
experiment and is recorded as such in `resolved_config.json`.

## Safety

* the loss is **unchanged** from the full run — changing it here would destroy
  the comparison this experiment exists to make;
* writes go to a separate artifact directory; the completed
  `e4_phobert_w2ner_full_v1` artifact is refused as a write target;
* `internal_test` is never opened; no organizer inference; no `output.zip`;
* logs carry counts, offsets and label ids only — never clinical text.

## Run-all order

1. mount Drive; 2. clone/update the repository; 3. repo root + `PYTHONPATH`;
4. resolve the governed train split by authoritative SHA-256; 5. select the
deterministic tiny subset; 6. gate on authorization; 7. acquire tokenizer and
encoder; 8. bounded training with periodic evaluation and heartbeats;
9. write the outcome.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import time

DRIVE_ROOT = Path("/content/drive/MyDrive/MedNorm-VI")
REPO_DIR = Path("/content/MedNorm-VI")
REPO_URL = os.environ.get("MEDNORM_REPO_URL", "https://github.com/vquclinh/MedNorm-VI")
REPO_REF = os.environ.get("MEDNORM_REPO_REF", "main")
CORPUS_DIR = DRIVE_ROOT / "data"
MODEL_CACHE_DIR = DRIVE_ROOT / "model_cache" / "huggingface"
VNCORENLP_DIR = DRIVE_ROOT / "model_cache" / "vncorenlp"

# ---------------------------------------------------------------------------
# OPERATOR SETTINGS
#
# Committed values are the DISABLED, NOT-AUTHORIZED state. A fresh Run all stops
# at the authorization gate and trains nothing.
#
# To run the diagnostic, an operator sets the run flag to True and pastes the
# authorization string exactly as it appears in
# configs/training/phase2_e4_tiny_overfit_diagnostic.yaml.
# ---------------------------------------------------------------------------
RUN_TINY_OVERFIT_DIAGNOSTIC = False
CONFIRM_TINY_OVERFIT = ""

TARGET_EXAMPLES = 12
SEED = 20260728
LEARNING_RATE = 5e-5
MAX_GRAD_NORM = 1.0

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount("/content/drive")
except Exception as error:  # noqa: BLE001 - Colab-only import
    print(json.dumps({"stage": "drive_mount_skipped", "detail": str(error)}))

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--prune"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True).stdout.strip()
print(json.dumps({"stage": "bootstrap", "git_commit": GIT_COMMIT}, indent=2, sort_keys=True))


In [ ]:
from mednorm_vi.mention_factory.w2ner import (
    W2NERLabelVocab,
    build_relation_grid_head,
)
from mednorm_vi.training.phase2.e4_collapse_diagnosis import load_governed_examples
from mednorm_vi.training.phase2.e4_tiny_overfit import (
    TINY_OVERFIT_ARTIFACT_NAME,
    TINY_OVERFIT_AUTHORIZATION,
    TINY_OVERFIT_EVALUATE_EVERY_N_EPOCHS,
    TINY_OVERFIT_MAX_EPOCHS,
    TINY_OVERFIT_TARGET_EXACT_F1,
    assert_artifact_dir_is_not_protected,
    assert_tiny_overfit_authorized,
    build_tiny_overfit_resolved_config,
    build_tiny_overfit_targets,
    score_predicted_grid,
    select_tiny_overfit_examples,
    should_stop_tiny_overfit,
    summarize_tiny_overfit,
)
from mednorm_vi.training.phase2.e4_w2ner_training import (
    E4_GOVERNED_TRAIN_SHA256,
    E4_MODEL_ID,
    E4_PINNED_MODEL_REVISION,
    atomic_relation_head_input_dim,
    build_atomic_projection,
    build_w2ner_batch_contract_from_segmented_words,
    decode_w2ner_logits,
    prepare_phobert_word_inputs,
    project_to_atomic_word_embeddings,
    resolve_governed_split_by_sha256,
    resolve_phobert_weight_format,
)
from mednorm_vi.training.phobert_alignment import (
    map_segmented_words,
    resolve_segmented_text,
    segmented_text_to_words,
)

# The diagnostic writes to its OWN directory. The completed full artifact is
# immutable evidence and this call refuses it outright.
OUTPUT_DIR = assert_artifact_dir_is_not_protected(
    DRIVE_ROOT / "artifacts" / TINY_OVERFIT_ARTIFACT_NAME)
print(json.dumps({
    "stage": "diagnostic_target",
    "output_dir": str(OUTPUT_DIR),
    "authorization_required": TINY_OVERFIT_AUTHORIZATION,
    "run_flag": RUN_TINY_OVERFIT_DIAGNOSTIC,
    "internal_test_accessed": False,
}, indent=2, sort_keys=True))


In [ ]:
# ---------------------------------------------------------------------------
# Resolve the governed TRAIN split by authoritative SHA-256, never by filename.
# internal_test is never resolved, opened or copied by this notebook.
# ---------------------------------------------------------------------------
TRAIN_RESOLUTION = resolve_governed_split_by_sha256(
    split="train",
    expected_sha256=E4_GOVERNED_TRAIN_SHA256,
    search_roots=(CORPUS_DIR, CORPUS_DIR / "processed",
                  DRIVE_ROOT / "data" / "derived", REPO_DIR / "data"),
)
TRAIN_SPLIT_PATH = TRAIN_RESOLUTION.path

SELECTION = select_tiny_overfit_examples(
    load_governed_examples(TRAIN_SPLIT_PATH, split="train"),
    split="train",
    target_size=TARGET_EXAMPLES,
)
if SELECTION.missing_required_types:
    print(json.dumps({
        "stage": "tiny_overfit_selection_warning",
        "missing_required_types": list(SELECTION.missing_required_types),
        "detail": "the governed corpus has no examples of these types",
    }, indent=2, sort_keys=True))
print(json.dumps(SELECTION.as_dict(), indent=2, sort_keys=True))

SELECTED_ROWS = set(SELECTION.row_indices)
EXAMPLES = [
    example for example in load_governed_examples(TRAIN_SPLIT_PATH, split="train")
    if example.row_index in SELECTED_ROWS
]
if len(EXAMPLES) != SELECTION.example_count:
    raise AssertionError("selected rows could not be re-read deterministically")


In [ ]:
# ---------------------------------------------------------------------------
# AUTHORIZATION GATE. A fresh Run all stops here and trains nothing.
# ---------------------------------------------------------------------------
try:
    assert_tiny_overfit_authorized(
        CONFIRM_TINY_OVERFIT, enabled=RUN_TINY_OVERFIT_DIAGNOSTIC)
    AUTHORIZED = True
except Exception as error:  # noqa: BLE001 - the disabled state is the default
    AUTHORIZED = False
    print(json.dumps({
        "stage": "tiny_overfit_not_authorized",
        "detail": str(error),
        "training_performed": False,
    }, indent=2, sort_keys=True))

if not AUTHORIZED:
    raise SystemExit(
        "E4 tiny-overfit diagnostic is not authorized; nothing was trained")


In [ ]:
import torch
from torch import nn
from transformers import AutoModel, AutoTokenizer

import py_vncorenlp  # noqa: F401 - imported for the segmenter side effect

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "logs").mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(MODEL_CACHE_DIR)

PINNED_MODEL_REVISION = os.environ.get(
    "MEDNORM_E4_MODEL_REVISION", "").strip() or E4_PINNED_MODEL_REVISION
WEIGHT_FORMAT = resolve_phobert_weight_format(E4_MODEL_ID, PINNED_MODEL_REVISION)

tokenizer = AutoTokenizer.from_pretrained(
    E4_MODEL_ID, revision=PINNED_MODEL_REVISION, cache_dir=str(MODEL_CACHE_DIR))
py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
SEGMENTER = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORENLP_DIR))

def segment_with_vncorenlp(text: str) -> str:
    return " ".join(SEGMENTER.word_segment(text))

VOCAB = W2NERLabelVocab()
MAX_WORDS = 256
MAX_MODEL_TOKENS = 256

def build_item(example):
    """One contract per selected example. Built once and reused every epoch."""
    segmented_text, _source = resolve_segmented_text(example.text, segment_with_vncorenlp)
    model_words = map_segmented_words(
        example.text, segmented_text_to_words(segmented_text))
    contract = build_w2ner_batch_contract_from_segmented_words(
        example.document_id, example.text, example.entities, model_words,
        max_words=MAX_WORDS, vocab=VOCAB)
    targets, gold = build_tiny_overfit_targets(example, vocab=VOCAB)
    if targets != contract.grid.labels:
        raise AssertionError("diagnostic target grid disagrees with the E4 contract")
    return contract, targets, gold

ITEMS = [build_item(example) for example in EXAMPLES]
print(json.dumps({
    "stage": "tiny_overfit_contracts_built",
    "examples": len(ITEMS),
    "atomic_words": [len(item[0].grid.words) for item in ITEMS],
    "gold_mentions": sum(len(item[2]) for item in ITEMS),
}, indent=2, sort_keys=True))

base_model = AutoModel.from_pretrained(
    E4_MODEL_ID, revision=PINNED_MODEL_REVISION, cache_dir=str(MODEL_CACHE_DIR),
    use_safetensors=WEIGHT_FORMAT.use_safetensors).to(DEVICE)
head = build_relation_grid_head(
    atomic_relation_head_input_dim(base_model.config.hidden_size),
    len(VOCAB.labels)).to(DEVICE)
trainable = list(base_model.parameters()) + list(head.parameters())
optimizer = torch.optim.AdamW(trainable, lr=LEARNING_RATE, weight_decay=0.0)

RESOLVED_CONFIG = build_tiny_overfit_resolved_config(
    selection=SELECTION,
    model_revision=PINNED_MODEL_REVISION,
    tokenizer_revision=PINNED_MODEL_REVISION,
    seed=SEED,
    learning_rate=LEARNING_RATE,
    precision_mode="fp32",
    device_type=DEVICE.type,
)
(OUTPUT_DIR / "resolved_config.json").write_text(
    json.dumps(RESOLVED_CONFIG, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(OUTPUT_DIR / "selection.json").write_text(
    json.dumps(SELECTION.as_dict(), indent=2, sort_keys=True) + "\n", encoding="utf-8")


In [ ]:
def word_embeddings_for(contract):
    encoding = prepare_phobert_word_inputs(
        tokenizer, contract.segmented_words, max_length=MAX_MODEL_TOKENS)
    projection = build_atomic_projection(
        contract.grid.original_text, contract.segmented_words, encoding,
        atomic_words=contract.atomic_words)
    model_inputs = {
        key: torch.tensor([value], dtype=torch.long, device=DEVICE)
        for key, value in encoding.model_inputs.items()
    }
    outputs = base_model(**model_inputs)
    return project_to_atomic_word_embeddings(outputs.last_hidden_state[0], projection)

def evaluate(epoch: int, mean_loss: float):
    """Score the SAME examples that were trained on. That is the experiment."""
    base_model.eval()
    head.eval()
    target_grids, predicted_grids = [], []
    gold_sets, predicted_sets = [], []
    with torch.no_grad():
        for contract, targets, gold in ITEMS:
            embeddings = word_embeddings_for(contract)
            pair_mask = torch.tensor(
                [contract.grid.pair_mask], dtype=torch.bool, device=DEVICE)
            logits = head(embeddings, pair_mask)[0].detach().cpu().tolist()
            size = len(contract.grid.words)
            predicted_grids.append(tuple(
                tuple(int(max(range(len(logits[row][col])),
                              key=lambda index: logits[row][col][index]))
                      for col in range(size))
                for row in range(size)))
            target_grids.append(targets)
            gold_sets.append(gold)
            predicted_sets.append(set(decode_w2ner_logits(contract, logits)))
    return score_predicted_grid(
        epoch=epoch, mean_training_loss=mean_loss,
        target_grids=target_grids, predicted_grids=predicted_grids,
        gold_mention_sets=gold_sets, predicted_mention_sets=predicted_sets,
        background_label_id=VOCAB.none_id)

HISTORY_PATH = OUTPUT_DIR / "logs" / "tiny_overfit_history.jsonl"
HISTORY_PATH.write_text("", encoding="utf-8")
scores = []
stopped_reason = "reached_max_epochs_without_target_exact_f1"
started = time.monotonic()

for epoch in range(1, TINY_OVERFIT_MAX_EPOCHS + 1):
    base_model.train()
    head.train()
    epoch_loss = 0.0
    for contract, targets, _gold in ITEMS:
        optimizer.zero_grad(set_to_none=True)
        embeddings = word_embeddings_for(contract)
        pair_mask = torch.tensor(
            [contract.grid.pair_mask], dtype=torch.bool, device=DEVICE)
        labels = torch.tensor([targets], dtype=torch.long, device=DEVICE)
        logits = head(embeddings, pair_mask)
        # The SAME loss the full run used. Changing it here would destroy the
        # comparison this diagnostic exists to make.
        loss = nn.functional.cross_entropy(
            logits.reshape(-1, len(VOCAB.labels)), labels.reshape(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, MAX_GRAD_NORM)
        optimizer.step()
        epoch_loss += float(loss.detach().float().cpu())
    mean_loss = epoch_loss / len(ITEMS)

    due = (epoch % TINY_OVERFIT_EVALUATE_EVERY_N_EPOCHS == 0
           or epoch == 1 or epoch == TINY_OVERFIT_MAX_EPOCHS)
    if not due:
        continue
    score = evaluate(epoch, mean_loss)
    scores.append(score)
    record = {
        **score.as_dict(),
        "elapsed_seconds": round(time.monotonic() - started, 3),
    }
    with HISTORY_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, sort_keys=True) + "\n")
    # Heartbeat: counts only, never text.
    print(json.dumps(record, indent=2, sort_keys=True))
    stop, reason = should_stop_tiny_overfit(
        epoch=epoch, exact_f1=score.exact_f1,
        target_exact_f1=TINY_OVERFIT_TARGET_EXACT_F1,
        max_epochs=TINY_OVERFIT_MAX_EPOCHS)
    if stop:
        stopped_reason = reason
        break

OUTCOME = summarize_tiny_overfit(scores, stopped_reason=stopped_reason)
(OUTPUT_DIR / "outcome.json").write_text(
    json.dumps(OUTCOME.as_dict(), indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(json.dumps(OUTCOME.as_dict(), indent=2, sort_keys=True))


## Return-to-repository

Record in the next append-only audit:

* `selection.json` — the deterministic row indices, document ids and entity
  counts (no clinical text);
* the final row of `logs/tiny_overfit_history.jsonl` — grid cell accuracy,
  **positive cell accuracy** and exact F1, reported separately;
* `outcome.json` — `pipeline_can_memorize` and its `interpretation`.

Do not report this as a quality result and do not initialize a full E4 run from
anything produced here. If `pipeline_can_memorize` is true, the remaining work is
the class-imbalance and corpus-ordering evidence recorded in Audit 0043; if it is
false with a near-zero loss, the target/loss/decoder inconsistency it exposes is
the thing to fix first.
